# core

> Minimal IPython notebook integration for an LLM `%%prompt` cell magic. Inspired by fast.ai's solveit.

The whole module is built around three observations:

1. An `.ipynb` is just JSON. Cells have sources, outputs, and (since nbformat 4.5) stable ids.
2. JupyterLab tells the kernel which cell is executing via `parent_header.metadata.cellId`. That's enough to locate ourselves in the notebook without text matching.
3. Cell outputs persist in the JSON on disk. If a prompt cell already has output, re-running it shouldn't cost tokens.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from IPython.display import Markdown, display
from IPython.core.magic import register_cell_magic

from pathlib import Path
import argparse, json, os, shlex

## System prompt

Kept short and honest about the actual environment. The model takes its tone, structure, and capabilities cues from here, so lies about features we don't have (tool use, variable injection, etc.) leak into responses.

In [ ]:
#| export
system_prompt = """You are an AI assistant embedded inside an IPython notebook through a `%%prompt` cell magic. The notebook is a linear sequence of markdown, code, and prompt cells executed in a single persistent kernel — state from code cells carries forward. When the user runs a prompt cell, every cell above it (sources and captured outputs) is sent to you as the conversation history; previous prompt cells appear as your prior assistant turns. The dialog *is* the notebook.

Be concise, direct, and incremental. Match response length to the question. Do not pad, restate, or end with "let me know if...". Use fenced code blocks with language tags. Default to idiomatic Python — comprehensions, broadcasting, fastcore-style brevity. Short single-line docstrings; no inline comments unless a constraint is genuinely non-obvious.

Your knowledge cutoff is January 2026."""

## Locating the current notebook

Two-step strategy. JupyterLab ≥ 3.5 sets `JPY_SESSION_NAME` to the notebook's path; use it to determine the current notebook's path.

In [ ]:
#| export
def _current_notebook():
    "Resolve the path of the notebook this kernel is serving."
    name = os.environ.get("JPY_SESSION_NAME")
    if name and Path(name).exists(): return Path(name)
    raise RuntimeError("Cannot determine current notebook; save the file or set JPY_SESSION_NAME")

In [ ]:
#| eval: false
_current_notebook()

Path('/Users/pablo/src/nbdialog/nbs/00_core.ipynb')

## Cells → chat messages

Prompt cells contribute one `user` message (the prompt body with the `%%prompt` line stripped) and, when they already have rendered output, one `assistant` message replaying that output. Code and markdown cells contribute their source as `user`, plus any captured outputs as a follow-up `user` message. The cell whose id matches `up_to_id` is the boundary — included for its prompt but never for its (stale) cached response, since that response is what the current call is producing.

In [ ]:
#| export
def _join(x): return "".join(x) if isinstance(x, list) else x

In [ ]:
#| export
def _is_prompt(cell):
    "True if `cell` is a code cell whose source starts with the %%prompt magic."
    if cell.get("cell_type") != "code": return False
    return _join(cell.get("source", "")).lstrip().startswith("%%prompt")

In [ ]:
#| export
def _strip_magic(src):
    "Drop the leading %%prompt line from a prompt cell's source."
    return "\n".join(_join(src).splitlines()[1:])

In [ ]:
#| export
def _output_text(out):
    "Extract a readable string from one nbformat output entry (for context, not response)."
    if out.get("output_type") == "stream": return _join(out.get("text", []))
    data = out.get("data", {})
    if "text/markdown" in data: return _join(data["text/markdown"])
    if "text/plain"    in data: return _join(data["text/plain"])
    return ""

def _response_markdown(cell):
    "Return the model's rendered markdown response stored in this cell's outputs, or None."
    for o in cell.get("outputs", []):
        data = o.get("data", {})
        if "text/markdown" in data: return _join(data["text/markdown"])
    return None

In [ ]:
#| export
def notebook_to_messages(cells, up_to_id, system=None):
    "Build chat messages from `cells` up to and including the cell with id `up_to_id`."
    msgs = [{"role": "system", "content": system or system_prompt}]
    for c in cells:
        src = _join(c["source"])
        is_current = c.get("id") == up_to_id
        if _is_prompt(c):
            msgs.append({"role": "user", "content": _strip_magic(src)})
            if not is_current:
                resp = _response_markdown(c)
                if resp: msgs.append({"role": "assistant", "content": resp})
        else:
            msgs.append({"role": "user", "content": src})
            for o in c.get("outputs", []):
                txt = _output_text(o)
                if txt: msgs.append({"role": "user", "content": f"# Output:\n{txt}"})
        if is_current: break
    return msgs

Sanity check on the demo notebook:

In [ ]:
demo = json.loads(Path("99_small_demo.ipynb").read_text())["cells"]
last = demo[-1]["id"]
[(m["role"], m["content"][:60]) for m in notebook_to_messages(demo, last)]

[('system', 'You are an AI assistant embedded inside an IPython notebook '),
 ('user', '# Small demo\n> Small test subject used to understand and tes'),
 ('user', 'from nbdialog.core import *\nfrom nbdialog.providers.azure im'),
 ('user', 'write me hello world in python, like a pirate!'),
 ('assistant', '```python\nprint("Ahoy, world!")\n```'),
 ('user', 'def hello():\n    print("Hello matey!")\n\nhello()'),
 ('user', '# Output:\nHello matey!\n'),
 ('user', 'which is better?'),
 ('assistant',
  'Yours, for pirate flavor.\n\n- Mine is the classic minimal “he'),
 ('user', 'Give me top 5 news about bitcoin as of May 12 2026')]

## Providers & tools

The magic doesn't know or care which vendor answers the prompt — it only needs a callable that turns messages into text. We encode that contract as a `typing.Protocol` rather than an ABC so a provider can be any duck-typed object the user happens to have, without inheriting from anything in this package. The optional `tools` parameter lets a provider, if it supports tool calling, run a request/response loop with the model and dispatch tool invocations against Python callables.

Tools are paired (schema, function) values: an OpenAI-shaped JSON tool envelope plus the Python implementation. We deliberately keep the schema explicit and hand-written for now — generating it from type hints/docstrings is a nice future addition, not a hard requirement.

Both registries are module-level singletons because a notebook kernel *is* a single global session — there is nothing for a context manager or thread-local to scope. Users register both once at the top of their notebook:

```python
from nbdialog.providers.azure import AzureProvider
from nbdialog.tools.search import web_search
set_provider(AzureProvider())
set_tools([web_search])
```

In [ ]:
#| export
from typing import Protocol, runtime_checkable

@runtime_checkable
class Provider(Protocol):
    "A vendor-agnostic chat-completion contract: messages in, text out."
    def complete(self, messages: list[dict], tools: list = None) -> str: ...

In [ ]:
#| export
from typing import NamedTuple, Callable

class Tool(NamedTuple):
    "A schema/function pair the model can call. `schema` is an OpenAI-shaped tool envelope."
    schema: dict
    fn: Callable

_provider: "Provider | None" = None
_tools: list[Tool] = []

def set_provider(p: Provider) -> None:
    "Register `p` as the provider the `%%prompt` magic will call."
    global _provider
    _provider = p

def get_provider() -> Provider:
    "Return the active provider, or raise with the fix in the message."
    if _provider is None:
        raise RuntimeError("Call nbdialog.set_provider(AzureProvider()) first.")
    return _provider

def set_tools(tools: list[Tool]) -> None:
    "Register the `Tool`s available on every `%%prompt`."
    global _tools
    _tools = list(tools)

def get_tools() -> list[Tool]:
    "Return the currently registered tools."
    return _tools

## The `%%prompt` magic

Flow on every execution:

1. Parse the magic line (`-f` / `--force`).
2. Read this cell's id from `parent_header.metadata.cellId`.
3. If the on-disk copy of the cell already has rendered output and `--force` was not passed, replay that output and stop — no API call.
4. Otherwise build the message list from all cells up to and including this one, call the model, and display the response.

The cache lives in the `.ipynb` file itself: outputs are persisted by the normal Jupyter save flow, so they survive kernel restarts and Run-All-from-top, which is the whole point of this feature.

In [ ]:
#| export
def _parse_prompt_args(line):
    "Parse the magic line, e.g. `%%prompt -f`."
    p = argparse.ArgumentParser(prog="%%prompt", add_help=False)
    p.add_argument("-f", "--force", action="store_true",
                   help="bypass cache and call the model even if this cell has output")
    return p.parse_args(shlex.split(line or ""))

In [ ]:
#| export
@register_cell_magic
def prompt(line, cell):
    "Send the notebook-so-far to the LLM and render its reply as markdown."
    args = _parse_prompt_args(line)
    ip = get_ipython()
    cell_id = ip.parent_header.get("metadata", {}).get("cellId")
    if not cell_id:
        raise RuntimeError("No cellId in parent_header — requires JupyterLab ≥ 3.5")

    nb_path = _current_notebook()
    cells = json.loads(nb_path.read_text())["cells"]
    current = next((c for c in cells if c.get("id") == cell_id), None)

    if not args.force and current:
        cached = _response_markdown(current)
        if cached:
            display(Markdown(cached))
            return

    messages = notebook_to_messages(cells, up_to_id=cell_id)
    display(Markdown(get_provider().complete(messages, tools=get_tools() or None)))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()